In [ ]:
# Use %autoreload 1 (explicit reload only) to avoid UnicodeDecodeError on Windows:
# %autoreload 2 reads stdlib sources with cp1252, which fails on UTF-8 files.
# %load_ext autoreload
# %autoreload 1
from napari import Viewer
import numpy as np
import matplotlib.pyplot as plt
import matplotlib

# matplotlib.rcParams["font.family"] = "DejaVu Sans"  # or "Arial", "Tahoma", etc.
from napari_pecan_py.widgets import color_tuner

viewer = Viewer()
viewer.window.add_plugin_dock_widget("napari-pecan-py", "Color Tuner")
viewer.window.add_plugin_dock_widget("napari-pecan-py", "Color Adjustments")
# viewer.window.add_plugin_dock_widget("napari-pecan-py", "Mask Retouching")
# viewer.window.add_plugin_dock_widget("napari-pecan-py", "YOLO Segmentation")

# viewer.window.add_plugin_dock_widget("napari-pecan-py", "Contrastive Coding")

In [ ]:
import os
from pathlib import Path
import tempfile

import cv2
import numpy as np
import torch

print("--- CUDA / torch ---")
print("torch:", torch.__version__)
print("cuda available:", torch.cuda.is_available())
if torch.cuda.is_available():
    print("cuda device_count:", torch.cuda.device_count())
    for i in range(torch.cuda.device_count()):
        print(f"cuda:{i}:", torch.cuda.get_device_name(i))

# -----------------------------
# Paths (sample_data only)
# -----------------------------
WORKSPACE = Path(r"c:\\Users\\salir\\Desktop\\pecan_py_napari")
DATA_DIR = WORKSPACE / "sample_data"

VIDEO_PATH = DATA_DIR / "GH013558-cropped.MP4"
CRACK_NPY = DATA_DIR / "GH013558-cropped.MP4 - Crack.npy"
KERNEL_NPY = DATA_DIR / "GH013558-cropped.MP4 - Kernel.npy"
PECAN_NPY = DATA_DIR / "GH013558-cropped.MP4 - Pecan.npy"

# -----------------------------
# Small dataset knobs
# -----------------------------
# Keep this small to iterate quickly.
max_frames = 60
imgsz = 256
epochs = 5
batch = 2
lr0 = 1e-3

# -----------------------------
# Load frames from MP4
# -----------------------------
cap = cv2.VideoCapture(str(VIDEO_PATH))
if not cap.isOpened():
    raise RuntimeError(f"Failed to open video: {VIDEO_PATH}")

fps = cap.get(cv2.CAP_PROP_FPS)
W = int(cap.get(cv2.CAP_PROP_FRAME_WIDTH))
H = int(cap.get(cv2.CAP_PROP_FRAME_HEIGHT))
N = int(cap.get(cv2.CAP_PROP_FRAME_COUNT))
print("--- Video ---")
print(f"name={VIDEO_PATH.name} frames≈{N} size={W}x{H} fps={fps}")

frames = []
while True:
    ok, frame_bgr = cap.read()
    if not ok:
        break
    frame_rgb = cv2.cvtColor(frame_bgr, cv2.COLOR_BGR2RGB)
    frames.append(frame_rgb)
cap.release()

frames = np.stack(frames, axis=0)  # (T,H,W,3)
T = frames.shape[0]
print("Loaded frames:", frames.shape, frames.dtype)

if T > max_frames:
    frames = frames[:max_frames]
T = frames.shape[0]
print("Using T:", T)


# -----------------------------
# Load masks from .npy
# -----------------------------
def normalize_mask(mask: np.ndarray, T: int, H: int, W: int, name: str) -> np.ndarray:
    m = np.asarray(mask)

    # Common shapes we might see: (T,H,W), (H,W), (T,H,W,1)
    if m.ndim == 2:
        m = np.broadcast_to(m[None, ...], (T, H, W))
    elif m.ndim == 3:
        pass
    elif m.ndim == 4:
        # (T,H,W,1) or similar
        m = m[..., 0]
    else:
        raise ValueError(f"Unexpected mask ndim for {name}: {m.shape} (ndim={m.ndim})")

    # Ensure T matches
    if m.shape[0] != T:
        m = m[:T]

    # Ensure H/W match
    if m.shape[1] != H or m.shape[2] != W:
        m2 = np.zeros((T, H, W), dtype=m.dtype)
        for t in range(T):
            m2[t] = cv2.resize(m[t].astype(np.uint8), (W, H), interpolation=cv2.INTER_NEAREST)
        m = m2

    return m


crack = np.load(CRACK_NPY)
kernel = np.load(KERNEL_NPY)
pecan = np.load(PECAN_NPY)

print("--- Masks (raw) ---")
print("crack:", crack.shape, crack.dtype)
print("kernel:", kernel.shape, kernel.dtype)
print("pecan:", pecan.shape, pecan.dtype)

crack = normalize_mask(crack, T=T, H=H, W=W, name="crack")
kernel = normalize_mask(kernel, T=T, H=H, W=W, name="kernel")
pecan = normalize_mask(pecan, T=T, H=H, W=W, name="pecan")

masks_by_class = {"Crack": crack, "Kernel": kernel, "Pecan": pecan}

# -----------------------------
# Export to YOLO-seg polygon dataset
# -----------------------------
from napari_pecan_py.widgets.yolo_seg.model import export_napari_seg_dataset

with tempfile.TemporaryDirectory(prefix="pecan_yolo_seg_nb_") as tmpdir:
    spec = export_napari_seg_dataset(frames, masks_by_class, tmpdir)

    print("--- Exported YOLO dataset ---")
    print("root:", spec.root)
    print("class_names:", spec.class_names)
    print("data_yaml:", spec.data_yaml)

    # -----------------------------
    # Train (CUDA + logs)
    # -----------------------------
    from ultralytics import YOLO

    device = "0" if torch.cuda.is_available() else "cpu"
    print("--- Training ---")
    print("device:", device)
    print(f"epochs={epochs} batch={batch} lr0={lr0} imgsz={imgsz}")

    model = YOLO("yolov8n-seg.pt")

    # Some ultralytics versions may not support `plots=`.
    try:
        results = model.train(data=str(spec.data_yaml), epochs=epochs, batch=batch, lr0=lr0, device=device, imgsz=imgsz, workers=0, verbose=True, plots=False)
    except TypeError as e:
        print("Retrying training without `plots=` due to TypeError:")
        print(e)
        results = model.train(data=str(spec.data_yaml), epochs=epochs, batch=batch, lr0=lr0, device=device, imgsz=imgsz, workers=0, verbose=True)

    # Ultralytics returns a Results object; best weights should be available on it.
    best = getattr(results, "best", None)
    print("--- Done ---")
    print("best weights:", best)

In [ ]:
# Standalone OpenCV workflow (no napari required)
from __future__ import annotations

from dataclasses import dataclass
from pathlib import Path
from typing import Iterable

import cv2
import numpy as np
import pandas as pd


@dataclass
class Swatch:
    idx: int
    bbox: tuple[int, int, int, int]
    rgb: tuple[int, int, int]
    hsv: tuple[int, int, int]


def _cluster_boxes_by_y(boxes: list[tuple[int, int, int, int]], y_tol: int = 20) -> list[list[tuple[int, int, int, int]]]:
    groups: list[list[tuple[int, int, int, int]]] = []
    for box in sorted(boxes, key=lambda b: b[1]):
        x, y, w, h = box
        cy = y + h // 2
        matched = False
        for grp in groups:
            gcy = int(np.mean([gy + gh // 2 for _, gy, _, gh in grp]))
            if abs(cy - gcy) <= y_tol:
                grp.append(box)
                matched = True
                break
        if not matched:
            groups.append([box])
    return groups


def detect_swatch_boxes(image_bgr: np.ndarray) -> list[tuple[int, int, int, int]]:
    hsv = cv2.cvtColor(image_bgr, cv2.COLOR_BGR2HSV)
    h, w = hsv.shape[:2]

    # Calibration strip is near the bottom of the card, not in cherry rows.
    bottom_band = np.zeros((h, w), dtype=np.uint8)
    y_start = int(h * 0.72)
    bottom_band[y_start:, :] = 255

    red1 = cv2.inRange(hsv, (0, 45, 35), (20, 255, 255))
    red2 = cv2.inRange(hsv, (160, 45, 35), (179, 255, 255))
    # Include magenta/purple tones that appear in this calibration strip.
    purple = cv2.inRange(hsv, (125, 35, 30), (179, 255, 255))
    dark = cv2.inRange(hsv, (0, 0, 0), (179, 90, 90))

    mask = cv2.bitwise_or(cv2.bitwise_or(red1, red2), cv2.bitwise_or(purple, dark))
    mask = cv2.bitwise_and(mask, bottom_band)
    mask = cv2.morphologyEx(mask, cv2.MORPH_OPEN, np.ones((3, 3), np.uint8), iterations=1)
    mask = cv2.morphologyEx(mask, cv2.MORPH_CLOSE, np.ones((5, 5), np.uint8), iterations=1)

    contours, _ = cv2.findContours(mask, cv2.RETR_EXTERNAL, cv2.CHAIN_APPROX_SIMPLE)
    candidate_boxes: list[tuple[int, int, int, int]] = []
    for c in contours:
        x, y, bw, bh = cv2.boundingRect(c)
        area = bw * bh
        if area < 120 or area > 1200:
            continue
        aspect = bw / float(bh)
        if 0.70 <= aspect <= 1.35 and 12 <= bw <= 36 and 12 <= bh <= 36:
            candidate_boxes.append((x, y, bw, bh))

    if not candidate_boxes:
        return []

    groups = _cluster_boxes_by_y(candidate_boxes, y_tol=18)
    groups = sorted(groups, key=lambda g: (len(g), np.mean([gy for _, gy, _, _ in g]), -np.std([gy for _, gy, _, _ in g]) if len(g) > 1 else 0), reverse=True)
    swatch_group = sorted(groups[0], key=lambda b: b[0])
    return swatch_group[:7]


def _plus_template(size: int, thickness: int) -> np.ndarray:
    tpl = np.zeros((size, size), dtype=np.uint8)
    c = size // 2
    cv2.line(tpl, (c, 1), (c, size - 2), 255, thickness=thickness)
    cv2.line(tpl, (1, c), (size - 2, c), 255, thickness=thickness)
    return tpl


def detect_plus_signs(image_bgr: np.ndarray, swatches: list[tuple[int, int, int, int]]) -> list[tuple[int, int, float]]:
    if not swatches:
        return []

    min_x = min(x for x, _, _, _ in swatches)
    max_x = max(x + w for x, _, w, _ in swatches)
    min_y = min(y for _, y, _, _ in swatches)
    max_y = max(y + h for _, y, _, h in swatches)

    gray = cv2.cvtColor(image_bgr, cv2.COLOR_BGR2GRAY)
    blackhat = cv2.morphologyEx(gray, cv2.MORPH_BLACKHAT, np.ones((9, 9), np.uint8))
    proc = cv2.GaussianBlur(blackhat, (3, 3), 0)

    anchors = [(min_x - 18, min_y - 12), (min_x - 18, max_y + 12), (max_x + 20, (min_y + max_y) // 2)]

    out: list[tuple[int, int, float]] = []
    for ax, ay in anchors:
        best = (ax, ay, -1.0)
        for size in (11, 15, 19, 23):
            half = 26
            x0 = max(ax - half, 0)
            y0 = max(ay - half, image_bgr.shape[0] // 2)
            x1 = min(ax + half, image_bgr.shape[1] - 1)
            y1 = min(ay + half, image_bgr.shape[0] - 1)
            patch = proc[y0 : y1 + 1, x0 : x1 + 1]
            if patch.shape[0] < size + 2 or patch.shape[1] < size + 2:
                continue

            tpl = _plus_template(size=size, thickness=max(1, size // 7))
            resp = cv2.matchTemplate(patch, tpl, cv2.TM_CCOEFF_NORMED)
            _, maxv, _, maxloc = cv2.minMaxLoc(resp)
            cx = x0 + maxloc[0] + size // 2
            cy = y0 + maxloc[1] + size // 2
            if maxv > best[2]:
                best = (int(cx), int(cy), float(maxv))
        out.append(best)

    return sorted(out, key=lambda d: (d[1], d[0]))


def sample_color_stats(image_bgr: np.ndarray, swatches: list[tuple[int, int, int, int]]) -> list[Swatch]:
    hsv = cv2.cvtColor(image_bgr, cv2.COLOR_BGR2HSV)
    out: list[Swatch] = []
    for i, (x, y, w, h) in enumerate(sorted(swatches, key=lambda b: b[0]), start=1):
        pad_x = max(1, int(w * 0.2))
        pad_y = max(1, int(h * 0.2))
        x0, x1 = x + pad_x, x + w - pad_x
        y0, y1 = y + pad_y, y + h - pad_y
        patch_bgr = image_bgr[y0:y1, x0:x1]
        patch_hsv = hsv[y0:y1, x0:x1]

        mean_bgr = np.mean(patch_bgr.reshape(-1, 3), axis=0)
        mean_hsv = np.mean(patch_hsv.reshape(-1, 3), axis=0)
        rgb = tuple(int(round(v)) for v in mean_bgr[::-1])
        hsv_val = tuple(int(round(v)) for v in mean_hsv)
        out.append(Swatch(idx=i, bbox=(x, y, w, h), rgb=rgb, hsv=hsv_val))
    return out


def annotate(image_bgr: np.ndarray, swatches: list[tuple[int, int, int, int]], pluses: list[tuple[int, int, float]]) -> np.ndarray:
    vis = image_bgr.copy()
    for i, (x, y, w, h) in enumerate(swatches, start=1):
        cv2.rectangle(vis, (x, y), (x + w, y + h), (0, 255, 255), 2)
        cv2.putText(vis, f"S{i}", (x, y - 6), cv2.FONT_HERSHEY_SIMPLEX, 0.5, (0, 255, 255), 1, cv2.LINE_AA)
    for i, (cx, cy, score) in enumerate(pluses, start=1):
        cv2.drawMarker(vis, (int(cx), int(cy)), (0, 255, 0), markerType=cv2.MARKER_CROSS, markerSize=20, thickness=2)
        cv2.putText(vis, f"+{i} {score:.2f}", (int(cx) + 8, int(cy) - 8), cv2.FONT_HERSHEY_SIMPLEX, 0.45, (0, 255, 0), 1, cv2.LINE_AA)
    return vis

In [ ]:
# Interactive UI for directory/image browsing and per-image results
import matplotlib.pyplot as plt
import ipywidgets as widgets
from IPython.display import display, clear_output


IMG_EXTS = {".png", ".jpg", ".jpeg", ".bmp", ".tif", ".tiff", ".webp"}


def list_images(folder: Path):
    folder = Path(folder)
    if not folder.exists() or not folder.is_dir():
        return []
    return sorted([p for p in folder.iterdir() if p.is_file() and p.suffix.lower() in IMG_EXTS])


def show_result(image_path: Path, out_dir: Path | None = None):
    result = run_detection(image_path)

    print(f"Image: {image_path}")
    print(f"Detected plus signs: {len(result['pluses'])}")
    print(f"Detected swatches: {len(result['swatches'])}")

    if not result["plus_df"].empty:
        display(result["plus_df"])
    else:
        print("No plus signs detected.")

    if not result["swatch_df"].empty:
        display(result["swatch_df"][["swatch", "bbox", "RGB", "HSV", "R", "G", "B", "H", "S", "V"]])
    else:
        print("No swatches detected.")

    rgb_img = cv2.cvtColor(result["annotated_bgr"], cv2.COLOR_BGR2RGB)
    plt.figure(figsize=(12, 8))
    plt.imshow(rgb_img)
    plt.title(image_path.name)
    plt.axis("off")
    plt.show()

    if out_dir is not None:
        out_dir.mkdir(parents=True, exist_ok=True)
        out_file = out_dir / f"{image_path.stem}_annotated.png"
        cv2.imwrite(str(out_file), result["annotated_bgr"])
        print(f"Saved annotated image: {out_file}")


dir_text = widgets.Text(value=str(Path.cwd() / "test"), description="Folder:", layout=widgets.Layout(width="75%"))
refresh_btn = widgets.Button(description="Refresh", button_style="info")
image_dropdown = widgets.Dropdown(options=[], description="Image:", layout=widgets.Layout(width="75%"))
prev_btn = widgets.Button(description="Previous")
next_btn = widgets.Button(description="Next")
run_btn = widgets.Button(description="Run Detection", button_style="success")
save_check = widgets.Checkbox(value=True, description="Save annotated")
out_text = widgets.Text(value=str(Path.cwd() / "test" / "_annotated"), description="Out:", layout=widgets.Layout(width="75%"))
output = widgets.Output()


image_list: list[Path] = []


def refresh_images(_=None):
    global image_list
    folder = Path(dir_text.value)
    image_list = list_images(folder)
    if image_list:
        image_dropdown.options = [(p.name, str(p)) for p in image_list]
        image_dropdown.value = str(image_list[0])
    else:
        image_dropdown.options = []
        image_dropdown.value = None
    with output:
        clear_output(wait=True)
        print(f"Found {len(image_list)} images in {folder}")


def shift_image(step: int):
    if not image_list or image_dropdown.value is None:
        return
    cur = Path(image_dropdown.value)
    idx = image_list.index(cur)
    new_idx = (idx + step) % len(image_list)
    image_dropdown.value = str(image_list[new_idx])


def run_current(_=None):
    with output:
        clear_output(wait=True)
        if image_dropdown.value is None:
            print("No image selected.")
            return
        img_path = Path(image_dropdown.value)
        out_dir = Path(out_text.value) if save_check.value else None
        show_result(img_path, out_dir=out_dir)


refresh_btn.on_click(refresh_images)
prev_btn.on_click(lambda _: shift_image(-1))
next_btn.on_click(lambda _: shift_image(1))
run_btn.on_click(run_current)

controls = widgets.VBox([widgets.HBox([dir_text, refresh_btn]), image_dropdown, widgets.HBox([prev_btn, next_btn, run_btn]), widgets.HBox([save_check, out_text])])

display(controls, output)
refresh_images()